# Yee Isentropic Vortex (2D)

This notebook runs the Yee isentropic-vortex benchmark: an analytic vortex perturbation on a uniform background, sampled on concentric shells (`sampleShellv2`) rather than a lattice. The outer `bufferRings` shells are held at their initial state by a Dirichlet boundary condition -- `sampleYeeVortex` returns that BC, which `yeeVortexCase.buildSystem` installs onto `schemeConfig.boundaryConditions` and applies once via `enforceDirichlet` before the run starts; the per-step enforcement after that is the scheme's own step function reading `schemeConfig.boundaryConditions`, not anything this notebook has to repeat.

Like `08-hydrostatic.ipynb`/`09-gresho-chan-vortex.ipynb`, this is a `particlePlot` (2D field view) case: plotting calls `buildFieldPlotter`/`refreshFieldPlotter` directly on `YEE_FIELDS` (exported from `warpSPH.cases.yeeVortex`) rather than going through `yeeVortexCase.setupPlot`/`updatePlot`, which do not live-update reliably inside a Jupyter cell in this environment.

`yeeVortexCase` has no `timestep` hook and leaves `dt` unset in its defaults, so `sampleYeeVortex` picks the CFL-derived `dt` during IC construction and the loop below is a fixed `range(nSteps)`.

Precision note: switching between single and double precision is controlled in the import/configuration cell below. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/10-Yee_Vortex.gif)


In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.yeeVortex import yeeVortexCase, YEE_FIELDS
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import torch
from tqdm.autonotebook import tqdm


In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on `10-yee-vortex.py`,
# made explicit and editable here. `yeeVortexCase.defaults`/`.params` are the
# same values the CLI script starts from -- anything not overridden below
# just keeps its case default.
spec = CaseSpec(caseName=yeeVortexCase.name, scheme=yeeVortexCase.scheme,
                params=dict(yeeVortexCase.params)) \
    .merged(**yeeVortexCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=200,
    dim=2,
    L=10.0,

    # --- time stepping ---------------------------------------------------
    tLimit=8.0,
    # No `dt` here -- `sampleYeeVortex` leaves the CFL-derived value from
    # `computeTimestep` in `config.dt`, and there is no `timestep` hook to
    # re-pick it, so it stays fixed for the whole run.

    # --- output --------------------------------------------------------------
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- Yee's own knobs (shell sampling + vortex/free-stream state) ---------
    params=dict(
        gamma=1.4,
        nr=32, bufferRings=10,
        xc=0.0, yc=0.0, beta=5.0,
        P_infty=1.0, rho_infty=1.0,
    ),
)
spec


In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`yeeVortexCase.buildSystem` -> `sampleYeeVortex` + `enforceDirichlet`),
# not re-derived here.
ctx = buildContext(yeeVortexCase, spec)
yeeVortexCase.configureScheme(ctx)
system = yeeVortexCase.buildSystem(ctx)
runningState = system.initializeNewState()


In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(YEE_FIELDS), not yeeVortexCase.setupPlot -- see
# the intro cell for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, YEE_FIELDS)

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = yeeVortexCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=yeeVortexCase.extraFields)


In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = yeeVortexCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, YEE_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=yeeVortexCase.extraFields)


In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)
